# Project - AI Invoice Processing Assistant

We'll now bring together what we've learned to make an AI Invoice Processing assistant for Accounts Payable

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "invoices.db"

OpenAI API Key exists and begins sk-proj-


In [30]:
system_message = """
You are a helpful assistant for an Accounts Payable team.
Give short, courteous answers.
Always be accurate. If you don't know the answer, say so.
Use your tools when you need information about purchase orders or invoices.

When processing a new invoice:
1. Check whether the invoice is a duplicate.
2. Validate the invoice against its purchase order.
3. Only save the invoice if it is not a duplicate and it matches the purchase order.
4. If there is a mismatch, explain the issue and do not save the invoice.
"""

## Purchase Order Database

We'll create a simple purchase order database that our Accounts Payable assistant can use.

In [31]:
# Create a simple purchase order database

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS purchase_orders (
            po_number TEXT PRIMARY KEY,
            vendor TEXT,
            amount REAL
        )
    ''')

    purchase_orders = [
        ("PO1001", "Microsoft", 2500),
        ("PO1002", "Dell", 4800),
        ("PO1003", "Adobe", 1200),
        ("PO1004", "Amazon Web Services", 3500)
    ]

    cursor.executemany(
        'INSERT OR REPLACE INTO purchase_orders (po_number, vendor, amount) VALUES (?, ?, ?)',
        purchase_orders
    )

    conn.commit()

In [32]:
def get_purchase_order(po_number):
    print(f"DATABASE TOOL CALLED: Getting purchase order {po_number}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'SELECT vendor, amount FROM purchase_orders WHERE po_number = ?',
            (po_number.upper(),)
        )
        result = cursor.fetchone()
        return f"Purchase order {po_number} is for {result[0]} with an amount of ${result[1]}" if result else "No purchase order data available"

In [33]:
get_purchase_order("PO1002")

DATABASE TOOL CALLED: Getting purchase order PO1002


'Purchase order PO1002 is for Dell with an amount of $4800.0'

## Invoice Processing

Now we'll add invoices to our database. Our assistant will be able to check for duplicate invoices, validate them against purchase orders, save new invoices and retrieve their status.

In [34]:
# Create the invoices table

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS invoices (
            invoice_number TEXT PRIMARY KEY,
            vendor TEXT,
            po_number TEXT,
            amount REAL,
            status TEXT
        )
    ''')
    conn.commit()

In [35]:
def check_duplicate_invoice(invoice_number):
    print(f"DATABASE TOOL CALLED: Checking invoice {invoice_number}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'SELECT invoice_number FROM invoices WHERE invoice_number = ?',
            (invoice_number.upper(),)
        )
        result = cursor.fetchone()
        return f"Invoice {invoice_number} already exists" if result else f"Invoice {invoice_number} is not a duplicate"

In [36]:
def validate_invoice(invoice_number, vendor, po_number, amount):
    print(f"DATABASE TOOL CALLED: Validating invoice {invoice_number} against {po_number}", flush=True)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'SELECT vendor, amount FROM purchase_orders WHERE po_number = ?',
            (po_number.upper(),)
        )
        result = cursor.fetchone()

        if not result:
            return f"Purchase order {po_number} was not found"

        po_vendor, po_amount = result

        if vendor.lower() != po_vendor.lower():
            return f"Invoice {invoice_number} has a vendor mismatch. Invoice vendor is {vendor}, but purchase order vendor is {po_vendor}"

        if float(amount) != float(po_amount):
            return f"Invoice {invoice_number} has an amount mismatch. Invoice amount is ${amount}, but purchase order amount is ${po_amount}"

        return f"Invoice {invoice_number} matches purchase order {po_number}"

In [37]:
def save_invoice(invoice_number, vendor, po_number, amount):
    print(f"DATABASE TOOL CALLED: Saving invoice {invoice_number}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO invoices (invoice_number, vendor, po_number, amount, status) VALUES (?, ?, ?, ?, ?)',
            (invoice_number.upper(), vendor, po_number.upper(), amount, "Pending Review")
        )
        conn.commit()
        return f"Invoice {invoice_number} has been saved with status Pending Review"

In [38]:
def get_invoice_status(invoice_number):
    print(f"DATABASE TOOL CALLED: Getting status for invoice {invoice_number}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'SELECT vendor, po_number, amount, status FROM invoices WHERE invoice_number = ?',
            (invoice_number.upper(),)
        )
        result = cursor.fetchone()
        return f"Invoice {invoice_number} for {result[0]} against {result[1]} is ${result[2]} and has status {result[3]}" if result else "No invoice data available"

### Testing our invoice functions

In [39]:
check_duplicate_invoice("INV1001")

DATABASE TOOL CALLED: Checking invoice INV1001


'Invoice INV1001 already exists'

In [43]:
get_invoice_status("INV1001")

DATABASE TOOL CALLED: Getting status for invoice INV1001


'Invoice INV1001 for Dell against PO1002 is $4800.0 and has status Pending Review'

In [44]:
validate_invoice("INV2001", "Microsoft", "PO1001", 2500)

DATABASE TOOL CALLED: Validating invoice INV2001 against PO1001


'Invoice INV2001 matches purchase order PO1001'

In [45]:
validate_invoice("INV2002", "Microsoft", "PO1001", 3900)

DATABASE TOOL CALLED: Validating invoice INV2002 against PO1001


'Invoice INV2002 has an amount mismatch. Invoice amount is $3900, but purchase order amount is $2500.0'

## Tool Calling

Now we'll give our AI assistant tools that can work with purchase orders and invoices.

In [46]:
purchase_order_function = {
    "name": "get_purchase_order",
    "description": "Get the vendor and amount for a purchase order.",
    "parameters": {
        "type": "object",
        "properties": {
            "po_number": {
                "type": "string",
                "description": "The purchase order number, for example PO1001",
            },
        },
        "required": ["po_number"],
        "additionalProperties": False
    }
}

In [47]:
duplicate_invoice_function = {
    "name": "check_duplicate_invoice",
    "description": "Check whether an invoice already exists in the invoice database.",
    "parameters": {
        "type": "object",
        "properties": {
            "invoice_number": {
                "type": "string",
                "description": "The invoice number, for example INV1001",
            },
        },
        "required": ["invoice_number"],
        "additionalProperties": False
    }
}

In [48]:
validate_invoice_function = {
    "name": "validate_invoice",
    "description": "Validate an invoice against its purchase order by checking that the vendor and amount match.",
    "parameters": {
        "type": "object",
        "properties": {
            "invoice_number": {
                "type": "string",
                "description": "The invoice number",
            },
            "vendor": {
                "type": "string",
                "description": "The vendor on the invoice",
            },
            "po_number": {
                "type": "string",
                "description": "The purchase order number on the invoice",
            },
            "amount": {
                "type": "number",
                "description": "The total amount on the invoice",
            },
        },
        "required": ["invoice_number", "vendor", "po_number", "amount"],
        "additionalProperties": False
    }
}

In [49]:
save_invoice_function = {
    "name": "save_invoice",
    "description": "Save a new invoice to the invoice database with a Pending Review status.",
    "parameters": {
        "type": "object",
        "properties": {
            "invoice_number": {
                "type": "string",
                "description": "The invoice number",
            },
            "vendor": {
                "type": "string",
                "description": "The name of the vendor",
            },
            "po_number": {
                "type": "string",
                "description": "The purchase order number associated with the invoice",
            },
            "amount": {
                "type": "number",
                "description": "The total amount of the invoice",
            },
        },
        "required": ["invoice_number", "vendor", "po_number", "amount"],
        "additionalProperties": False
    }
}

In [50]:
invoice_status_function = {
    "name": "get_invoice_status",
    "description": "Get the current status and details of an invoice.",
    "parameters": {
        "type": "object",
        "properties": {
            "invoice_number": {
                "type": "string",
                "description": "The invoice number, for example INV1001",
            },
        },
        "required": ["invoice_number"],
        "additionalProperties": False
    }
}

In [51]:
tools = [
    {"type": "function", "function": purchase_order_function},
    {"type": "function", "function": duplicate_invoice_function},
    {"type": "function", "function": validate_invoice_function},
    {"type": "function", "function": save_invoice_function},
    {"type": "function", "function": invoice_status_function}
]

tools

[{'type': 'function',
  'function': {'name': 'get_purchase_order',
   'description': 'Get the vendor and amount for a purchase order.',
   'parameters': {'type': 'object',
    'properties': {'po_number': {'type': 'string',
      'description': 'The purchase order number, for example PO1001'}},
    'required': ['po_number'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'check_duplicate_invoice',
   'description': 'Check whether an invoice already exists in the invoice database.',
   'parameters': {'type': 'object',
    'properties': {'invoice_number': {'type': 'string',
      'description': 'The invoice number, for example INV1001'}},
    'required': ['invoice_number'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'validate_invoice',
   'description': 'Validate an invoice against its purchase order by checking that the vendor and amount match.',
   'parameters': {'type': 'object',
    'properties': {'invoice_num

In [52]:
def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:

        if tool_call.function.name == "get_purchase_order":
            arguments = json.loads(tool_call.function.arguments)
            po_number = arguments.get("po_number")
            result = get_purchase_order(po_number)

        elif tool_call.function.name == "check_duplicate_invoice":
            arguments = json.loads(tool_call.function.arguments)
            invoice_number = arguments.get("invoice_number")
            result = check_duplicate_invoice(invoice_number)

        elif tool_call.function.name == "validate_invoice":
            arguments = json.loads(tool_call.function.arguments)
            invoice_number = arguments.get("invoice_number")
            vendor = arguments.get("vendor")
            po_number = arguments.get("po_number")
            amount = arguments.get("amount")
            result = validate_invoice(invoice_number, vendor, po_number, amount)

        elif tool_call.function.name == "save_invoice":
            arguments = json.loads(tool_call.function.arguments)
            invoice_number = arguments.get("invoice_number")
            vendor = arguments.get("vendor")
            po_number = arguments.get("po_number")
            amount = arguments.get("amount")
            result = save_invoice(invoice_number, vendor, po_number, amount)

        elif tool_call.function.name == "get_invoice_status":
            arguments = json.loads(tool_call.function.arguments)
            invoice_number = arguments.get("invoice_number")
            result = get_invoice_status(invoice_number)

        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })

    return responses

In [53]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content

## Gradio

Now we'll create a simple Gradio interface to test our Accounts Payable assistant.

In [54]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting purchase order PO1004
DATABASE TOOL CALLED: Getting status for invoice INV1001
DATABASE TOOL CALLED: Checking invoice INV4001
DATABASE TOOL CALLED: Validating invoice INV4001 against PO1003
DATABASE TOOL CALLED: Saving invoice INV4001
DATABASE TOOL CALLED: Getting status for invoice INV4001
DATABASE TOOL CALLED: Checking invoice INV4002
DATABASE TOOL CALLED: Validating invoice INV4002 against PO1003
DATABASE TOOL CALLED: Checking invoice INV4003
DATABASE TOOL CALLED: Getting purchase order PO1003


## Conclusion

1. An AI Accounts Payable assistant for invoice processing
2. Tool calling with purchase order and invoice database lookups
3. Duplicate invoice detection and purchase order validation
4. A step towards an Agentic Accounts Payable workflow